# A practical tutorial for PATSTAT: From queries to user interfaces 

## Tutorial objectives
By the end of this tutorial, you will be able to:
- **Connect to a database**: Establish a connection to the PATSTAT database using Python.
- **Perform common searches**: Execute three key types of patent searches: by applicant, by IPC-based technology field, and by key owrd.
- **Write effective SQL**: Construct multi-table `JOIN` queries tailored to the PATSTAT schema.
- **Build user interfaces**: Create three different user interfaces (UIs) using `ipywidgets`, custom `CSS/HTML`, and `ipyvuetify`.
- **Export data**: Save your search results for further analysis.

## What makes this tutorial educational?
This notebook is structured for clear, step-by-step learning:
1.  **Cell-by-cell explanations**: Every code cell is introduced by a markdown cell explaining its purpose, the logic it contains, and why it is needed.
2.  **Real-world data fetching**: You will learn to query and display human-readable technology field descriptions directly from PATSTAT tables, a practical skill for analysis.
3.  **Modular learning**: We will first focus on each search type in isolation before combining them into a single, flexible function.
4.  **Comparing UI frameworks**: We will build three UIs to demonstrate the strengths of different tools for various tasks. A **UI (User Interface) framework** is a toolkit of pre-made components that helps developers build user-friendly graphical interfaces efficiently.

**Note on classification**: This tutorial uses the **International Patent Classification (IPC)** system and the 35 high-level technology fields from WIPO, which are based on IPC codes. This provides a strategic view of technological innovation.

**Platform**: This notebook uses Google BigQuery SQL syntax.

## Step 1: Setting up the environment

Our first step is to prepare our environment. This involves importing all the necessary Python libraries in one organised location and then establishing a live connection to the PATSTAT database.

### 1.1 - Importing libraries

This cell handles all library imports. We follow the best practice of importing everything at the start of the notebook. This makes dependencies clear and helps avoiding errors. We will import libraries for data manipulation (`pandas`), database interaction (`sqlalchemy`), and our three user interface (UI) frameworks.

In [1]:
# Import essential data and system libraries
import pandas as pd
from datetime import datetime
import warnings
import json

# Suppress common warnings for a cleaner output
warnings.filterwarnings("ignore")

# Import database connection libraries
from sqlalchemy import text
from epo.tipdata.patstat import PatstatClient

# UI Frameworks
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import ipyvuetify as v

print("All required libraries imported successfully.")

All required libraries imported successfully.


### 1.2 - Connecting to PATSTAT

With our libraries loaded, this cell establishes the connection to the PATSTAT database. It initialises a `PatstatClient`, creates a `session` for executing queries, and runs a simple `SELECT 1` query to verify that the connection is active and working correctly.

**Please note** that for the purpose of this training module we are connecting to the TEST environment representing a sub-set of the PATSTAT database. This is advised for developing code as it is faster and consumes less resources. To carry out analysis for production purposes, however, it is recommended to switch to the PROD environment by replacing "TEST" with "PROD" in the code.  

In [2]:
# Initialise connection variables to None for safe error handling
client = None
session = None

try:
    print("Connecting to PATSTAT database...")
    client = PatstatClient(env="TEST")
    session = client.orm()

    print("Successfully connected to PATSTAT!")
    print("Database Backend: Google BigQuery")

    # Test the connection to confirm it's live
    if session.execute(text("SELECT 1")).scalar() == 1:
        print("Connection test successful. Ready to query data.")

except Exception as e:
    print(f"❌ DATABASE CONNECTION FAILED: {e}")

Connecting to PATSTAT database...
Successfully connected to PATSTAT!
Database Backend: Google BigQuery
Connection test successful. Ready to query data.


## Step 2: Learning individual search types

To build a versatile tool, we must first understand its parts. In this section, we will develop three separate functions, each dedicated to a specific type of search. This modular approach is excellent for learning and debugging before we combine the functions.

### 2.1 — Search by applicant (name)

This cell defines `search_by_applicant`, a focused helper that retrieves **patent applications** filed by a given **applicant (e.g. a company or a university)** within a **date window**, returning a tidy DataFrame with the most relevant identifiers and metadata.

#### What the function returns (the DataFrame schema)

Each row is one application (after de-duping with `DISTINCT`) and includes the following headers:

* `appln_id` — The unique identifier for a patent application in PATSTAT.
* `filing_authority` — The two-letter code for the patent office or authority where the application was filed (e.g., `EP`, `US`).
* `filing_date` — The date on which the application was filed.
* `applicant_name` — The standardised name of the applicant (company or individual).

#### Which PATSTAT tables are involved and why

* `tls201_appln a` — **Application master table**. This is the starting point for any search, providing the core application details like application ID (`appln_id`), filing authority (`appln_auth`), and the filing date (`appln_filing_date`).
* `tls207_pers_appln pa` — **Person-Application link table**. This crucial table connects an application to a person (or entity). It contains `appln_id` and `person_id` and, most importantly, role indicators (`applt_seq_nr` for applicant, `invt_seq_nr` for inventor) that allow us to specify that we are only interested in applicants and not inventors.
* `tls206_person p` — **Person table**. This table stores the harmonised names of individuals and companies (`psn_name`) and is linked via `person_id`. We join to this table to search for a specific company name and to display it in the results.

#### How we ensure we are getting applicants (not inventors)

PATSTAT encodes roles via sequence numbers in `tls207_pers_appln`:

* `pa.applt_seq_nr > 0` → the person appears as an **applicant** (one or more applicants are usually numbered 1, 2, …).
* `pa.invt_seq_nr = 0` → the same link is **not** an inventor role.
  Together these clauses select **applicant links only**, excluding inventor-only records.

#### Applicant name matching (case-insensitive, partial)

* `UPPER(p.psn_name) LIKE :company_name` with `company_name` bound as `"%{company_name.upper()}%"`:

  * `UPPER(...)` + `LIKE` implements a **case-insensitive substring** match.
  * Using a **parameterized placeholder** (`:company_name`) avoids SQL injection.

#### Date filtering, sorting, and limiting

* `a.appln_filing_date BETWEEN :start_date AND :end_date` selects applications filed between the two dates **including** both dates.
* `ORDER BY a.appln_filing_date DESC` returns **newest filings first**—a common analyst preference.
* `LIMIT :limit` caps result size for responsiveness (especially in notebooks); bump this if you need more.

#### Why `DISTINCT`?

Because `tls207_pers_appln` can create **multiple link rows** per application (e.g., multiple applicants, or duplicated through different person harmonizations), `SELECT DISTINCT` ensures **one row per unique application + selected columns**, preventing duplicates in the output.

#### Joins in plain language

* `a` (applications) ⟶ join to `pa` (person ↔ application links) **by `appln_id`** to get role metadata.
* `pa` ⟶ join to `p` (persons) **by `person_id`** to read the applicant’s standardised name.

#### Why include the filing authority (`appln_auth`)

* Jurisdiction context: `EP` vs `US` vs `CN` influences **law, procedure, and coverage**.
* Downstream filters: analysts often **split or compare** by authority.
* Disambiguation: the same `appln_id` style can differ by dataset; `appln_auth` makes the record **self-describing**.

#### Parameters and safety

All user inputs (`company_name`, `start_date`, `end_date`, `limit`) are passed as **bind parameters** (`:param`) to:

* Prevent SQL injection (not really relevant on the TIP, but nice to learn anyways).
* Help the query planner cache and reuse execution plans.
* Keep Python string interpolation away from raw SQL.

#### Error handling + return shape

* If no `session`, we fail fast and return an **empty DataFrame**.
* On exceptions, we log the failure and also return an **empty DataFrame**—making the function safe in pipelines.

#### Practical notes & common variations

* **Name noise**: Corporate names change; consider searching for **canonical person IDs** or maintaining a **synonym list** for high-recall searches.
* **Applicants with multiple roles**: Some entities appear as both applicant and inventor on different filings; our filters ensure **applicant-only** rows here.
* **Pagination**: For UI use, pair `LIMIT` with an `OFFSET` or keyset pagination (e.g., `WHERE appln_filing_date < ?`) for stable paging.

In [3]:
def search_by_applicant(
    session,
    company_name: str,
    start_date: str = "2020-01-01",
    end_date: str = "2023-12-31",
    limit: int = 10,
):
    """Searches for patent applications by applicant name within a date range."""
    if not session:
        print("❌ No database connection available.")
        return pd.DataFrame()

    sql_query = text("""
        SELECT DISTINCT
            a.appln_id,
            a.appln_auth AS filing_authority,
            a.appln_filing_date AS filing_date,
            p.psn_name AS applicant_name
        FROM tls201_appln a
        JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
        JOIN tls206_person p ON pa.person_id = p.person_id
        WHERE
            pa.applt_seq_nr > 0
            AND pa.invt_seq_nr = 0
            AND a.appln_filing_date BETWEEN :start_date AND :end_date
            AND UPPER(p.psn_name) LIKE :company_name
        ORDER BY a.appln_filing_date DESC
        LIMIT :limit
    """)

    params = {
        "company_name": f"%{company_name.upper()}%",
        "start_date": start_date,
        "end_date": end_date,
        "limit": limit,
    }

    try:
        result = session.execute(sql_query, params)
        df = pd.DataFrame(result.fetchall(), columns=result.keys())
        print(f"✅ Found {len(df)} applications for applicant '{company_name}'.")
        return df
    except Exception as e:
        print(f"❌ Search failed: {e}")
        return pd.DataFrame()

#### Running a demo applicant search

Now we'll test our `search_by_applicant` function. This cell calls the function with sample data—searching for "SAMSUNG" patents applications between 2022 and 2023. If successful, it displays the resulting DataFrame and a simple analysis of the top filing authorities. This confirms our function works as expected.

In [4]:
print("=== DEMO: Searching for 'SAMSUNG' patents from 2022-01-01 to 2023-12-31 ===\n")
if session:
    df_applicant = search_by_applicant(
        session,
        company_name="SAMSUNG",
        start_date="2022-01-01",
        end_date="2023-12-31",
        limit=10
    )

    if not df_applicant.empty:
        print("\n📋 Search Results:")
        display(df_applicant)

        print("\n📊 Quick Analysis (Top Filing Authorities):")
        analysis_summary = pd.DataFrame({
            'Count': df_applicant['filing_authority'].value_counts().head(3)
        })
        display(analysis_summary)

=== DEMO: Searching for 'SAMSUNG' patents from 2022-01-01 to 2023-12-31 ===

✅ Found 10 applications for applicant 'SAMSUNG'.

📋 Search Results:


,appln_id,filing_authority,filing_date,applicant_name
0,601143381,WO,2023-03-15,SAMSUNG ELECTRONICS COMPANY
1,595892484,KR,2022-12-27,AGRICULTURAL CORPORATION SAMSUNG AGRO COMPANY
2,597176678,DE,2022-12-14,SAMSUNG ELECTRONICS COMPANY
3,588489031,US,2022-11-14,SAMSUNG ELECTRONICS COMPANY
4,595617160,WO,2022-11-02,SAMSUNG ELECTRONICS COMPANY
5,588914683,WO,2022-09-05,SAMSUNG ELECTRONICS COMPANY
6,585198725,US,2022-09-05,SAMSUNG ELECTRONICS COMPANY
7,583950561,US,2022-07-18,SAMSUNG DISPLAY
8,579125010,US,2022-05-23,SAMSUNG ELECTRONICS COMPANY
9,600176896,KR,2022-03-31,SAMSUNG HEAVY INDUSTRIES



📊 Quick Analysis (Top Filing Authorities):


,Count
filing_authority,
US,4
WO,3
KR,2


### 2.2 - Search by technology field (IPC-based)

Next, we will create a function to search for patents in a specific technology area using the 35 WIPO technology classification fields. This system groups IPC codes into fields like "Computer technology" or "Pharmaceuticals." To do this, we will query by joining the application table with `tls230_appln_techn_field` (which links an application to a field number) and `tls901_techn_field_ipc` (which contains the field label).

#### 2.2.1 - Discovering available technology fields

Before searching, it is helpful to know what terms are available. This next cell queries the PATSTAT database to retrieve a complete list of the 35 official WIPO technology fields and their corresponding sectors. The output is a pandas DataFrame that you can use as a reference for your searches in the interfaces below.

In [5]:
print("Fetching all available WIPO technology fields from the database...")

if session:
    try:
        # Use pandas read_sql for a convenient way to get the full table
        tech_field_query = text("""
            SELECT DISTINCT
                techn_sector,
                techn_field
            FROM tls901_techn_field_ipc
            ORDER BY techn_sector, techn_field
        """)
        
        df_fields = pd.read_sql(tech_field_query, session.connection())

        if not df_fields.empty:
            print(f"✅ Successfully retrieved {len(df_fields)} unique technology fields.")
            # Set pandas option to display all rows so none are truncated
            pd.set_option('display.max_rows', len(df_fields))
            display(df_fields)
            pd.reset_option('display.max_rows') # Reset option to default
        else:
            print("Could not retrieve the list of technology fields.")

    except Exception as e:
        print(f"❌ Failed to fetch technology fields: {e}")

Fetching all available WIPO technology fields from the database...
✅ Successfully retrieved 35 unique technology fields.


,techn_sector,techn_field
0,Chemistry,Basic materials chemistry
1,Chemistry,Biotechnology
2,Chemistry,Chemical engineering
3,Chemistry,Environmental technology
4,Chemistry,Food chemistry
5,Chemistry,"Macromolecular chemistry, polymers"
6,Chemistry,"Materials, metallurgy"
7,Chemistry,Micro-structural and nano-technology
8,Chemistry,Organic fine chemistry
9,Chemistry,Pharmaceuticals


#### 2.2.2 - Defining the technology field search function

This function, `search_by_technology_field`, retrieves patent applications based on the 35 high-level WIPO technology fields. This requires joining three tables to bridge the gap from an application to the human-readable field name.

##### What the function returns (the DataFrame schema)

Each row represents a unique patent application and includes the following headers:

* `appln_id` — The unique identifier for a patent application.
* `filing_authority` — The two-letter code for the patent office where the application was filed.
* `filing_date` — The date on which the application was filed.
* `techn_sector` — The broad technology sector (e.g., "Chemistry", "Electrical engineering").
* `techn_field` — The specific technology field within the sector (e.g., "Pharmaceuticals", "Computer technology").


Please note that in the table `tls230_appln_techn_field`, the attribute WEIGHT is often used to account for the relative contribution of each technological field to a single patent application (since one application can belong to multiple fields). To remain consistent with the educational purpose of this notebook, a simplified approach has been adopted.


##### Which PATSTAT tables are involved and why

* `tls201_appln a` — **Application master table**. As always, this is our starting point, containing the essential `appln_id` and `appln_filing_date` for filtering.
* `tls230_appln_techn_field tf` — **Application-technology link table**. This table connects an `appln_id` to a numerical technology field identifier, `techn_field_nr`. It serves as the bridge between the application and its classification.
* `tls901_techn_field_ipc ipc` — **Technology field lookup table**. This table contains the human-readable names for the technology fields. It maps the `techn_field_nr` from the previous table to descriptive text for both the broader sector (`techn_sector`) and the specific field (`techn_field`), allowing us to search using terms like "Computer technology".

##### Joins in plain language

1. Start with `tls201_appln` to access all applications.
2. `JOIN` to `tls230_appln_techn_field` on `appln_id` to find the technology field number for each application.
3. `JOIN` to `tls901_techn_field_ipc` on `techn_field_nr` to retrieve the text descriptions for those numbers, which we then use for filtering and display.

In [6]:
def search_by_technology_field(
    session,
    tech_field_keyword: str,
    start_date: str = "2020-01-01",
    end_date: str = "2023-12-31",
    limit: int = 10,
):
    """Searches for patents by a WIPO technology field keyword within a date range."""
    if not session:
        print("❌ No database connection available.")
        return pd.DataFrame()

    sql_query = text("""
        SELECT DISTINCT
            a.appln_id,
            a.appln_auth AS filing_authority,
            a.appln_filing_date AS filing_date,
            ipc.techn_sector,
            ipc.techn_field
        FROM tls201_appln a
        JOIN tls230_appln_techn_field tf ON a.appln_id = tf.appln_id
        JOIN tls901_techn_field_ipc ipc ON tf.techn_field_nr = ipc.techn_field_nr
        WHERE
            a.appln_filing_date BETWEEN :start_date AND :end_date
            AND (
                UPPER(ipc.techn_sector) LIKE :keyword 
                OR UPPER(ipc.techn_field) LIKE :keyword
            )
        ORDER BY a.appln_filing_date DESC
        LIMIT :limit
    """)

    params = {
        "keyword": f"%{tech_field_keyword.upper()}%",
        "start_date": start_date,
        "end_date": end_date,
        "limit": limit,
    }

    try:
        result = session.execute(sql_query, params)
        df = pd.DataFrame(result.fetchall(), columns=result.keys())
        print(f"✅ Retrieved {len(df)} applications for technology field '{tech_field_keyword}'.")
        return df
    except Exception as e:
        print(f"❌ Query failed: {e}")
        return pd.DataFrame()

#### Running a demo technology field search

This cell tests our `search_by_technology_field` function. We will search for patents in the "Computer technology" field filed in 2023. The output will show the results, including the broad technology sector and the specific technology field, demonstrating that our dynamic data fetching is working.

In [7]:
print("=== DEMO: Searching for patents in 'Computer technology' in 2023 ===\n")
if session:
    df_tech = search_by_technology_field(
        session,
        tech_field_keyword="Computer technology",
        start_date="2023-01-01",
        end_date="2023-12-31",
        limit=10
    )

    if not df_tech.empty:
        print("\n📋 Search results (with technology fields):")
        display(df_tech)
        
        print("\n📊 Quick analysis (Top filing authorities in this field):")
        analysis_summary_tech = pd.DataFrame({
            'Count': df_tech['filing_authority'].value_counts().head(3)
        })
        display(analysis_summary_tech)

=== DEMO: Searching for patents in 'Computer technology' in 2023 ===

✅ Retrieved 10 applications for technology field 'Computer technology'.

📋 Search results (with technology fields):


,appln_id,filing_authority,filing_date,techn_sector,techn_field
0,605263939,CN,2023-12-22,Electrical engineering,Computer technology
1,605275835,CN,2023-12-21,Electrical engineering,Computer technology
2,604981277,CN,2023-12-19,Electrical engineering,Computer technology
3,605117953,CN,2023-12-19,Electrical engineering,Computer technology
4,605102958,CN,2023-12-18,Electrical engineering,Computer technology
5,605099272,CN,2023-12-18,Electrical engineering,Computer technology
6,604977285,CN,2023-12-18,Electrical engineering,Computer technology
7,604859138,CN,2023-12-15,Electrical engineering,Computer technology
8,604875458,CN,2023-12-15,Electrical engineering,Computer technology
9,605102810,CN,2023-12-15,Electrical engineering,Computer technology



📊 Quick analysis (Top filing authorities in this field):


,Count
filing_authority,
CN,10


### 2.3 - Search by key word (title/abstract)

This cell defines our third search function, `search_by_text`. It finds patents containing keywords in their title or abstract. This is useful for finding new concepts that may not yet have a  classification symbol assigned. The function dynamically chooses which table to join (`tls202_appln_title` or `tls203_appln_abstr`) based on user input.

**Please note** that you may miss documents that do not contain the keywords you were searching with but might nevertheless be relevant. A proper search strategy combining different search criteria is vital to arrive at a solid result set. 

#### What the function returns (the DataFrame schema)

Each row is one unique patent application and includes the following headers:

* `appln_id` — The unique identifier for a patent application.
* `filing_authority` — The two-letter code for the patent office where the application was filed.
* `filing_date` — The date on which the application was filed.
* `applicant_name` — The standardised name of the applicant.
* `text_content` — The content of the patent title or abstract that was searched.

#### Which PATSTAT tables are involved and why

* `tls201_appln a` — **Application master table**. The required starting point for its `appln_id` and for date filtering.
* `tls202_appln_title t` — **Application title table**. This table contains the title of the patent application (`appln_title`), linked by `appln_id`. We join to this table when the user chooses to search within titles.
* `tls203_appln_abstr ab` — **Application abstract table**. Similarly, this table holds the abstract text (`appln_abstract`) and is used when the user wants to search within abstracts.
* `tls207_pers_appln` and `tls206_person` — These are included via `LEFT JOIN` to optionally fetch the applicant's name for display purposes. Unlike a filtering join, a `LEFT JOIN` ensures that we still get the patent record even if an applicant name is not found.

In [8]:
def search_by_text(
    session,
    keyword: str,
    field: str = "title",
    start_date: str = "2020-01-01",
    end_date: str = "2023-12-31",
    limit: int = 10,
):
    """Searches patent titles or abstracts for a keyword within a date range."""
    if not session:
        print("❌ No database connection.")
        return pd.DataFrame()

    if field == "title":
        text_table, text_column = "tls202_appln_title", "appln_title"
    elif field == "abstract":
        text_table, text_column = "tls203_appln_abstr", "appln_abstract"
    else:
        print(f"❌ Invalid field '{field}'. Choose 'title' or 'abstract'.")
        return pd.DataFrame()

    sql_query = text(f"""
        SELECT DISTINCT
            a.appln_id,
            a.appln_auth AS filing_authority,
            a.appln_filing_date AS filing_date,
            p.psn_name AS applicant_name,
            t.{text_column} AS text_content
        FROM tls201_appln a
        JOIN {text_table} t ON a.appln_id = t.appln_id
        LEFT JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id AND pa.applt_seq_nr > 0 AND pa.invt_seq_nr = 0
        LEFT JOIN tls206_person p ON pa.person_id = p.person_id
        WHERE a.appln_filing_date BETWEEN :start_date AND :end_date
          AND UPPER(t.{text_column}) LIKE :keyword
        ORDER BY a.appln_filing_date DESC
        LIMIT :limit
    """)

    params = {
        "keyword": f"%{keyword.upper()}%",
        "start_date": start_date,
        "end_date": end_date,
        "limit": limit,
    }

    try:
        result = session.execute(sql_query, params)
        df = pd.DataFrame(result.fetchall(), columns=result.keys())
        print(f"✅ Found {len(df)} applications with '{keyword}' in the {field}.")
        return df
    except Exception as e:
        print(f"❌ Search failed: {e}")
        return pd.DataFrame()

#### Running a demo text search

Finally, we test our `search_by_text` function. This cell searches for the phrase "neural network" in patent titles from 2023. We will truncate the long title text before displaying the results. This test ensures our text search capability is working correctly.

In [9]:
print("=== DEMO: Searching for 'neural network' in patent titles during 2023 ===\n")
if session:
    df_text = search_by_text(
        session,
        keyword="neural network",
        field="title",
        start_date="2023-01-01",
        end_date="2023-12-31",
        limit=10
    )

    if not df_text.empty:
        df_text['text_content'] = df_text['text_content'].str[:80] + '...'
        print("\n📋 Search Results (text content is truncated for display):")
        display(df_text)

=== DEMO: Searching for 'neural network' in patent titles during 2023 ===

✅ Found 10 applications with 'neural network' in the title.

📋 Search Results (text content is truncated for display):


,appln_id,filing_authority,filing_date,applicant_name,text_content
0,599440760,CN,2023-08-30,CHINA UNIVERSITY OF PETROLEUM (EAST CHINA),Soil humidity inversion method based on multi-...
1,602249940,CN,2023-08-29,YANSHAN UNIVERSITY,Subsynchronous oscillation source positioning ...
2,598761322,CN,2023-08-21,HUNAN UNIVERSITY OF SCIENCE AND TECHNOLOGY,Cabin wind speed correction method and system ...
3,598223733,CN,2023-06-14,"TECHNOLOGY TRANSFER CENTER CO., LTD., YANCHENG...",Bearing fault diagnosis method and system base...
4,598223733,CN,2023-06-14,YANCHENG INSTITUTE OF TECHNOLOGY,Bearing fault diagnosis method and system base...
5,595669251,CN,2023-06-13,SHANDONG UNIVERSITY,Fan blade impact positioning method and system...
6,598762750,CN,2023-06-07,SOUTHEAST UNIVERSITY,Dynamic reactive power compensation method bas...
7,598230451,CN,2023-05-24,ZHEJIANG UNIVERSITY,Industrial control system anomaly detection me...
8,598230451,CN,2023-05-24,HANGZHOU UWNTEK AUTOMATION SYSTEM COMPANY,Industrial control system anomaly detection me...
9,595375147,CN,2023-04-07,YANCHENG INSTITUTE OF TECHNOLOGY,Synchronous control method of second-order neu...


## Step 3: The combined search function

Now that we understand each search function, we can build the core of our application: `execute_combined_search`. This function will integrate all three search types, dynamically constructing an SQL query based on the user's input. This is the function our UIs will call to get data.

### 3.1 - Defining the combined search logic

This cell defines the `execute_combined_search` function. This is the most critical piece of logic in the tutorial. It has been carefully designed to be both flexible and robust, solving common challenges in building search tools.

- **Dynamic SQL**: It builds the filtering part of the query piece by piece, only adding `JOIN`s and `WHERE` conditions for filters that the user provides.
- **Robust subquery pattern (CTEs)**: It uses a multi-step query with Common Table Expressions (CTEs) to ensure stable and correct results. First, it finds the unique set of patent application IDs that match all filters. Then, it fetches the details for this definitive set. 
- **Consistent Output Schema**: The query is structured to always return the same set of columns, regardless of the search criteria. Columns for which no filter was applied will be present but contain null values. This is crucial for user interfaces that expect a fixed data structure.
- **SQL Window Functions**: To ensure exactly one row is returned per patent application (even if it has multiple applicants), it uses the `ROW_NUMBER()` function to pick the most relevant record for display, performing de-duplication directly within the database for maximum efficiency.

In [10]:
def execute_combined_search(
    applicant: str = "",
    tech_field: str = "",
    search_text: str = "",
    text_location: str = "title",
    start_date: str = "2020-01-01",
    end_date: str = "2023-12-31",
    limit: int = 20
):
    """Executes a combined patent search with a stable, multi-step CTE query and a consistent output schema."""
    if not session:
        return {"success": False, "data": pd.DataFrame(), "message": "No database connection"}

    try:
        # --- Part 1: Dynamically build the filtering clauses and parameters ---
        id_finding_joins = ["FROM tls201_appln a"]
        id_finding_where = ["a.appln_filing_date BETWEEN :start_date AND :end_date"]
        params = {"start_date": start_date, "end_date": end_date, "limit": limit}

        if applicant:
            id_finding_joins.append("JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id AND pa.applt_seq_nr > 0")
            id_finding_joins.append("JOIN tls206_person p ON pa.person_id = p.person_id")
            id_finding_where.append("UPPER(p.psn_name) LIKE :applicant")
            params["applicant"] = f"%{applicant.upper()}%"

        if tech_field:
            id_finding_joins.append("JOIN tls230_appln_techn_field tf ON a.appln_id = tf.appln_id")
            id_finding_joins.append("JOIN tls901_techn_field_ipc ipc ON tf.techn_field_nr = ipc.techn_field_nr")
            id_finding_where.append("(UPPER(ipc.techn_sector) LIKE :tech_field OR UPPER(ipc.techn_field) LIKE :tech_field)")
            params["tech_field"] = f"%{tech_field.upper()}%"
        
        if search_text:
            params["search_text"] = f"%{search_text.upper()}%"
            text_conditions = []
            if text_location in ["title", "both"]:
                id_finding_joins.append("LEFT JOIN tls202_appln_title t ON a.appln_id = t.appln_id")
                text_conditions.append("UPPER(t.appln_title) LIKE :search_text")
            if text_location in ["abstract", "both"]:
                id_finding_joins.append("LEFT JOIN tls203_appln_abstr ab ON a.appln_id = ab.appln_id")
                text_conditions.append("UPPER(ab.appln_abstract) LIKE :search_text")
            if text_conditions:
                id_finding_where.append(f"({' OR '.join(text_conditions)})")
        
        id_finding_joins = list(dict.fromkeys(id_finding_joins)) # De-duplicate joins for safety

        # --- Part 2: Construct the full query using Common Table Expressions (CTEs) for clarity and correctness ---
        final_query = f"""
        WITH matching_ids AS (
            -- Step 1: Find the unique, sorted, and limited set of application IDs that match all criteria.
            -- This is the crucial step that guarantees stability and correct narrowing of results.
            SELECT a.appln_id, MAX(a.appln_filing_date) as filing_date
            {' '.join(id_finding_joins)}
            WHERE {' AND '.join(id_finding_where)}
            GROUP BY a.appln_id
            ORDER BY filing_date DESC
            LIMIT :limit
        ),
        ranked_results AS (
            -- Step 2: Fetch all details for the found IDs and de-duplicate in one step.
            -- We use ROW_NUMBER() to pick only the first relevant record for each appln_id,
            -- ensuring a single row per patent even with multiple applicants or tech fields.
            SELECT 
                ids.appln_id,
                a.appln_auth AS filing_authority,
                a.appln_filing_date AS filing_date,
                p.psn_name AS applicant_name,
                ipc.techn_sector,
                ipc.techn_field,
                t.appln_title AS title,
                ab.appln_abstract AS abstract,
                ROW_NUMBER() OVER(PARTITION BY ids.appln_id ORDER BY pa.applt_seq_nr) as rn
            FROM matching_ids ids
            JOIN tls201_appln a ON ids.appln_id = a.appln_id
            LEFT JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id AND pa.applt_seq_nr > 0
            LEFT JOIN tls206_person p ON pa.person_id = p.person_id
            LEFT JOIN tls230_appln_techn_field tf ON a.appln_id = tf.appln_id
            LEFT JOIN tls901_techn_field_ipc ipc ON tf.techn_field_nr = ipc.techn_field_nr
            LEFT JOIN tls202_appln_title t ON a.appln_id = t.appln_id
            LEFT JOIN tls203_appln_abstr ab ON a.appln_id = ab.appln_id
        )
        -- Final Step: Select all columns for a consistent schema, taking only the de-duplicated rows.
        SELECT appln_id, filing_authority, filing_date, applicant_name, techn_sector, techn_field, title, abstract
        FROM ranked_results
        WHERE rn = 1
        ORDER BY filing_date DESC
        """

        result = session.execute(text(final_query), params)
        df = pd.DataFrame(result.fetchall(), columns=result.keys())

        message = f"Found {len(df)} unique patents matching your criteria."
        if df.empty:
            message = "No patents found. Try broader search criteria."
        
        return {"success": True, "data": df, "message": message}

    except Exception as e:
        return {"success": False, "data": pd.DataFrame(), "message": f"Search failed: {e}"}

## Step 4: Building interactive user interfaces

Now we will use our `execute_combined_search` function as the engine to power three different interactive user interfaces (UIs). This will demonstrate how a single backend logic can serve multiple front-end experiences and teach the strengths of each UI framework.

A key feature of these interfaces is their **dynamic nature**. The 'Search Location' options are initially hidden. They will only appear once you start typing in the 'Text Search' field, ensuring the UI remains clean and uncluttered.

### User interface #1 - Standard `ipywidgets`

`ipywidgets` is the default library for creating UI controls in Jupyter. It's simple, robust, and ideal for building internal tools and prototypes. We will create text boxes, sliders, and buttons, and then link them to Python functions that execute our search.

#### 4.1.1 - Defining the visibility toggle function for UI #1

This function, `toggle_text_location_w1`, is responsible for the dynamic behavior of the UI. It is connected to the 'Text Search' input field and will show the 'Search in' radio buttons when there is text, and hide them when the field is empty.

In [11]:
def toggle_text_location_w1(change):
    """Shows or hides the text location widget based on text input."""
    if change.new:
        w1_location.layout.display = 'flex' # 'flex' is the default display for ipywidgets
    else:
        w1_location.layout.display = 'none'

#### 4.1.2 - Defining the search handler for UI #1

This function, `w1_on_search`, will be triggered when the 'Search' button is clicked. It gathers values from all the input widgets, calls our main `execute_combined_search` function, and displays the results in the output area.

In [12]:
w1_results_df = None

def w1_on_search(button):
    """Event handler for the ipywidgets search button click."""
    global w1_results_df

    with w1_output:
        clear_output(wait=True)
        print("🔍 Searching patents...")
        result = execute_combined_search(
            applicant=w1_applicant.value, tech_field=w1_tech_field.value, search_text=w1_text.value,
            text_location=w1_location.value, limit=w1_limit.value
        )
        clear_output(wait=True)
        print(f"{result['message']}")
        if result["success"] and not result["data"].empty:
            w1_results_df = result["data"]
            display(w1_results_df)
            w1_export_btn.disabled = False
        else:
            w1_results_df = None
            w1_export_btn.disabled = True

#### 4.1.3 - Defining the export handler for UI #1

This `w1_on_export` function saves the latest search results to a CSV file when the 'Export' button is clicked. It is only enabled after a successful search.

In [13]:
def w1_on_export(button):
    """Event handler for the ipywidgets export button click."""
    if w1_results_df is not None:
        filename = f"ipywidgets_search_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        w1_results_df.to_csv(filename, index=False)
        with w1_output:
            print(f"\n💾 Exported results to: {filename}")

#### 4.1.4 - Building and assembling the `ipywidgets` interface

This cell constructs the visual part of our first UI. It defines the widgets, connects their events to the handler functions, and arranges them in a layout. Note that `w1_location` is initially hidden by setting its `layout.display` to `'none'`, and we use `.observe()` to link our toggle function to the text input field.

In [14]:
print("Building user interface #1: Standard ipywidgets...")

w1_applicant = widgets.Text(description="Company:", placeholder="e.g., Samsung")
w1_tech_field = widgets.Text(description="Tech Field:", placeholder="e.g., Computer technology")
w1_text = widgets.Text(description="Text Search:", placeholder="e.g., neural network")
w1_location = widgets.RadioButtons(options=["title", "abstract", "both"], value="title", description="Search in:", layout={'display': 'none'})
w1_limit = widgets.IntSlider(value=10, min=5, max=50, step=5, description="Max Results:")
w1_search_btn = widgets.Button(description="Search Patents", button_style="info", icon="search")
w1_export_btn = widgets.Button(description="Export CSV", button_style="success", icon="download", disabled=True)
w1_output = widgets.Output()

w1_text.observe(toggle_text_location_w1, names='value')
w1_search_btn.on_click(w1_on_search)
w1_export_btn.on_click(w1_on_export)

ipywidgets_interface = widgets.VBox([
    widgets.HTML("<h3>Interface #1: Standard ipywidgets</h3>"),
    w1_applicant, w1_tech_field, w1_text, w1_location, w1_limit,
    widgets.HBox([w1_search_btn, w1_export_btn]),
    w1_output
], layout=widgets.Layout(border='1px solid #ccc', padding='15px'))

print("✅ The interface is ready. Run the next cell to display it.")

Building user interface #1: Standard ipywidgets...
✅ The interface is ready. Run the next cell to display it.


#### 4.1.5 - Displaying the `ipywidgets` interface

This cell renders the `ipywidgets_interface` object. Running it will make the interactive UI appear in your notebook's output.

In [15]:
display(ipywidgets_interface)

### User interface #2 - A highly-styled ipywidgets UI

For a truly professional and custom look, this interface demonstrates an advanced technique. While it is built entirely with `ipywidgets`, we apply a large, custom block of CSS to transform the standard widgets into a polished, modern web application layout. This gives us the reliability of Python callbacks with the visual fidelity of a custom design.

#### 4.2.1 - Initialising the state variable

This cell initialises a global variable, `html_results_df`. This variable will hold the pandas DataFrame from the most recent successful search, making it accessible to other functions, like the one for exporting data.

In [16]:
html_results_df = None

#### 4.2.2 - Defining the CSS style helper

This cell defines a helper function, `h2_style_block`, which returns the entire CSS for our UI as a single string. Encapsulating the styles in a function keeps the main code cleaner. This CSS block targets the custom classes we will add to our widgets to create the two-column layout, the dark sidebar, and all other visual effects.

In [17]:
def h2_style_block():
    return """
    <style>
        :root {
            --sidebar-bg: #1f2937; --content-bg: #f9fafb; --card-bg: #ffffff;
            --border-color: #e5e7eb; --text-primary: #111827; --text-secondary: #6b7280;
            --accent-color: #3b82f6; --accent-color-dark: #2563eb;
        }

        /* App frame */
        .patstat-app-h2 {
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
            border: 1px solid var(--border-color);
            border-radius: 12px;
            box-shadow: 0 10px 30px rgba(0,0,0,0.1);
            width: 100%;
            max-width: 1300px; 
            margin: auto;
            box-sizing: border-box;
            overflow-x: hidden; /* kill any horizontal overflow at root */
        }

        /* Make sure all widget containers never force horizontal scroll */
        .patstat-app-h2 .widget-box,
        .patstat-app-h2 .widget-hbox,
        .patstat-app-h2 .widget-vbox,
        .patstat-app-h2 .jupyter-widgets,
        .patstat-app-h2 .widget-container {
            overflow-x: hidden !important;
        }
        /* Allow flex items to shrink instead of causing overflow */
        .patstat-app-h2 .widget-hbox > .widget-box,
        .patstat-app-h2 .widget-vbox > .widget-box {
            min-width: 0 !important;
        }

        .patstat-sidebar { box-sizing: border-box; flex-shrink: 0; }
        .patstat-sidebar h3 { color: #fff; font-size: 20px; margin-bottom: 24px; }
        .form-group label { display: block; font-size: 14px; font-weight: 500; margin-bottom: 8px; color: #d1d5db; }

        .patstat-app-h2 .widget-text input[type="text"] {
            width: 100%;
            border-radius: 6px;
            border: 1px solid #4b5563;
            background-color: #374151;
            color: #f9fafb;
            box-sizing: border-box;
        }
        .patstat-app-h2 .widget-text input::placeholder { color: #FFFFFF; }

        .patstat-app-h2 .widget-radio-buttons label { color: #d1d5db !important; }
        .patstat-app-h2 .widget-slider .widget-label { color: #d1d5db; }

        .btn { width: 100%; font-size: 16px; font-weight: 600; margin-top: 8px; }
        .btn-primary button { background-color: var(--accent-color) !important; color: #fff !important; }
        .btn-primary button:hover { background-color: var(--accent-color-dark) !important; }
        .btn-secondary button { background-color: #4b5563 !important; color: #d1d5db !important; }
        .btn button:disabled { cursor: not-allowed; opacity: 0.5; }

        .content-area-h2 {
            background-color: var(--content-bg);
            padding: 24px;
            min-height: 400px;
            width: 100%;
            max-width: 100%;
            box-sizing: border-box;
            overflow-x: hidden; /* no horizontal scroll in content area */
        }

        .initial-message { color: var(--text-secondary); text-align: center; margin-top: 50px; }
        .error-message { background: #ffebee; border: 1px solid #ef9a9a; border-radius: 8px; padding: 20px; text-align: center; color: #c62828; }

        .pro-header {
            background: #1f2937; color: #f9fafb;
            padding: 16px 24px; border-radius: 8px 8px 0 0;
            display: flex; align-items: center; gap: 12px; font-size: 18px; font-weight: 600;
        }
        .pro-analysis-grid {
            display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
            gap: 16px; padding: 24px; background: #f9fafb;
        }
        .pro-card { background: #fff; border: 1px solid #e5e7eb; border-radius: 8px; padding: 16px; box-shadow: 0 1px 2px rgba(0,0,0,0.05); transition: all 0.2s ease-in-out; }
        .pro-card:hover { transform: translateY(-4px); box-shadow: 0 4px 12px rgba(0,0,0,0.1); }
        .pro-card-title { font-size: 12px; font-weight: 600; color: #4b5563; margin-bottom: 8px; text-transform: uppercase; }
        .pro-card-value { font-size: 16px; font-weight: 600; color: #111827; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }

        /* Table: disable horizontal overflow and force wrapping */
        .pro-table-container {
            padding: 24px;
            background: #fff;
            border-radius: 0 0 8px 8px;
            max-width: 100%;
            overflow-x: hidden !important; /* override any earlier styles */
            overflow-y: auto;
            box-sizing: border-box;
        }
        .pro-table {
            width: 100%;
            border-collapse: collapse;
            table-layout: fixed;           /* prevent wide columns from blowing layout */
            word-wrap: break-word;
        }
        .pro-table th,
        .pro-table td {
            white-space: normal;           /* allow wrapping */
            overflow-wrap: anywhere;       /* modern wrap */
            word-break: break-word;        /* fallback */
            padding: 12px 16px;
            font-size: 14px;
        }
        .pro-table th {
            background: #f3f4f6; color: #374151;
            text-align: left; font-size: 12px; font-weight: 600;
            text-transform: uppercase; letter-spacing: 0.5px;
            border-bottom: 2px solid #e5e7eb;
        }
        .pro-table tr:hover td { background: #f9fafb; }
    </style>
    """

#### 4.2.3 - Defining the results HTML helper

This helper function, `h2_build_results_html`, is responsible for presentation. It takes a DataFrame of search results and transforms it into a polished HTML block, complete with a header, analysis cards, and a styled table. **It also truncates the long abstract text to ensure the UI remains clean.** This separates the data processing from the final visual rendering.

In [18]:
def h2_build_results_html(df):
    # Create a copy to avoid modifying the global dataframe used for export
    display_df = df.copy()

    # Helper function to truncate text to the first n words
    def truncate_text(text, num_words=6):
        if isinstance(text, str):
            words = text.split()
            if len(words) > num_words:
                return ' '.join(words[:num_words]) + '...'
            return text
        return text

    # Truncate the title and abstract columns if they exist
    if 'title' in display_df.columns:
        display_df['title'] = display_df['title'].apply(truncate_text)
    if 'abstract' in display_df.columns:
        display_df['abstract'] = display_df['abstract'].apply(truncate_text)
    
    # Drop columns that are entirely empty for a cleaner display in this specific UI
    display_df.dropna(axis=1, how='all', inplace=True)

    table_html = display_df.to_html(classes="pro-table", index=False, escape=False)
    countries = ", ".join(display_df["filing_authority"].value_counts().head(3).index)
    
    top_applicants = "N/A"
    if 'applicant_name' in display_df.columns and not display_df['applicant_name'].isnull().all():
        top_applicants = ", ".join([str(name)[:35] for name in display_df["applicant_name"].value_counts().head(3).index])

    header = "<div class='pro-header'><span>Search Results</span></div>"
    kpis = f"""
    <div class='pro-analysis-grid'>
        <div class='pro-card'>
            <div class='pro-card-title'>Records</div>
            <div class='pro-card-value'>{len(display_df):,}</div>
        </div>
        <div class='pro-card'>
            <div class='pro-card-title'>Top jurisdictions</div>
            <div class='pro-card-value' title="{countries}">{countries}</div>
        </div>
        <div class='pro-card'>
            <div class='pro-card-title'>Top applicants</div>
            <div class='pro-card-value' title="{top_applicants}">{top_applicants}</div>
        </div>
    </div>
    """
    table = f"<div class='pro-table-container'>{table_html}</div>"
    return f"<div class='content-area-h2'>{header}{kpis}{table}</div>"

#### 4.2.4 - Defining the visibility toggle handler

This function, `h2_toggle_location`, handles the dynamic visibility of the 'Search In' radio buttons. It is an event handler that listens for changes in the 'Text Search' input field.

In [19]:
def h2_toggle_location(change):
    # This function is attached to the `text_search` widget
    if change['name'] == 'value':
        h2_location.layout.display = '' if (change['new'] or '').strip() else 'none'

#### 4.2.5 - Defining the search handler

This is the primary event handler, `h2_on_search`, which is triggered when the 'Search' button is clicked. It orchestrates the entire search process: clearing previous output, showing a 'Searching...' message, calling the `execute_combined_search` function, and then using our helper to display the results or an error message.

In [20]:
def h2_on_search(button):
    global html_results_df
    # The with statement correctly targets the output widget
    with h2_output:
        # First, clear everything from the previous run
        h2_output.clear_output(wait=True)
        
        # Then, display the loading message. 
        display(HTML("<div class='content-area-h2'><div class='initial-message'>🔍 Searching...</div></div>"))
    
    try:
        res = execute_combined_search(
            h2_applicant.value, h2_tech_field.value, h2_text_search.value,
            h2_location.value, "2020-01-01", "2023-12-31", int(h2_limit.value)
        )
        # Target the output widget again for the final results
        with h2_output:
            h2_output.clear_output(wait=True)
            
            if res.get("success") and res.get("data") is not None and not res["data"].empty:
                html_results_df = res["data"]
                results_html = h2_build_results_html(res["data"])
                display(HTML(results_html))
                h2_btn_export.disabled = False
            else:
                html_results_df = None
                msg = res.get("message", "No results.")
                display(HTML(f"<div class='content-area-h2'><div class='error-message'>❌ {msg}</div></div>"))
                h2_btn_export.disabled = True
    except Exception as e:
        with h2_output:
            h2_output.clear_output(wait=True)
            
            err = str(e).replace("<", "&lt;").replace(">", "&gt;")
            display(HTML(f"<div class='content-area-h2'><div class='error-message'>❌ Python Error: {err}</div></div>"))
            h2_btn_export.disabled = True

#### 4.2.6 - Defining the export handler

This handler, `h2_on_export`, is connected to the 'Export CSV' button. It checks if there are results stored in the `html_results_df` global variable and, if so, saves them to a CSV file.

In [21]:
def h2_on_export(button):
    with h2_output:
        if html_results_df is None or html_results_df.empty:
            display(HTML("<div class='error-message'>❌ Nothing to export.</div>"))
            return
        
        try:
            path = f"patstat_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
            html_results_df.to_csv(path, index=False)
            display(HTML(f"<div class='initial-message'>✅ Exported to <code>{path}</code></div>"))
        except Exception as e:
            err = str(e).replace("<", "&lt;").replace(">", "&gt;")
            display(HTML(f"<div class='error-message'>❌ Export failed: {err}</div>"))

#### 4.2.7 - Building and displaying the ipywidgets-based interface

This is the main construction cell for UI #2. It performs several key steps:
1.  Defining all the individual `ipywidgets`.
2.  Adding custom CSS classes to the buttons for styling.
3.  Connecting the event handler functions to the appropriate widgets using `.observe()` and `.on_click()`.
4.  Assembling the entire layout using `VBox` and `HBox` containers to create the final two-column design.
5.  Finally, it injects the CSS styles and displays the fully composed UI.

In [22]:
print("Building user interface #2: Custom HTML/CSS with ipywidgets...")

# ---- Define all widgets ----
h2_applicant = widgets.Text(placeholder="e.g., Google", layout=widgets.Layout(width='100%'))
h2_tech_field = widgets.Text(placeholder="e.g., Computer technology", layout=widgets.Layout(width='100%'))
h2_text_search = widgets.Text(placeholder="e.g., neural network", layout=widgets.Layout(width='100%'))

h2_location = widgets.RadioButtons(options=[('Title', 'title'), ('Abstract', 'abstract'), ('Both', 'both')], value='title', layout=widgets.Layout(display='none'))
h2_limit = widgets.IntSlider(value=10, min=5, max=50, step=5)
h2_btn_search = widgets.Button(description="Search")
h2_btn_export = widgets.Button(description="Export CSV", disabled=True)
h2_output = widgets.Output()

# ---- Apply custom CSS classes for styling ----
h2_btn_search.add_class('btn'); h2_btn_search.add_class('btn-primary')
h2_btn_export.add_class('btn'); h2_btn_export.add_class('btn-secondary')

# ---- Connect event handlers ----
h2_text_search.observe(h2_toggle_location, names='value')
h2_btn_search.on_click(h2_on_search)
h2_btn_export.on_click(h2_on_export)

# ---- Assemble the UI layout ----
sidebar_content = widgets.VBox([
    widgets.HTML("<h3>Search Filters</h3>"),
    widgets.HTML("<label>Company</label>"), h2_applicant,
    widgets.HTML("<label>Technology Field</label>"), h2_tech_field,
    widgets.HTML("<label>Text Search</label>"), h2_text_search,
    widgets.HTML("<label>Search In</label>"), h2_location,
    widgets.HTML("<label>Max Results</label>"), h2_limit,
    h2_btn_search,
    h2_btn_export
], layout=widgets.Layout(padding='0 24px 24px 24px'))

# Sidebar (fixed width)
sidebar = widgets.VBox([sidebar_content],
    layout=widgets.Layout(width='380px', background='#1f2937')
)
sidebar.add_class('patstat-sidebar')

# Content (flexible, min_width=0 prevents overflow forcing a scrollbar)
content = widgets.VBox(
    [h2_output],
    layout=widgets.Layout(flex='1 1 0%', min_width='0')  # 0% avoids min-content overflow
)

# Final container (hide scrollbars)
ui2_container = widgets.HBox(
    [sidebar, content],
    layout=widgets.Layout(width='100%', overflow_x='hidden', overflow_y='visible')
)

ui2_app = widgets.VBox([ui2_container])
ui2_app.add_class('patstat-app-h2')

# --- Display the UI ---
display(HTML(h2_style_block()))
display(ui2_app)

with h2_output:
    display(HTML("<div class='content-area-h2'><div class='initial-message'>Your search results will appear here.</div></div>"))


Building user interface #2: Custom HTML/CSS with ipywidgets...


### User interface #3 - `ipyvuetify` (Material Design)

For the most modern, application-like experience, we use `ipyvuetify`, which brings Google's Material Design framework into Jupyter. It provides a rich set of beautiful and functional widgets, including a powerful `DataTable` for displaying results.

**Note:** This section requires `ipyvuetify` to be installed (`pip install ipyvuetify`).

#### 4.3.1 - Defining the visibility toggle function for UI #3

This `toggle_text_location_v3` function is connected to the 'Text Search' field's `input` event. It dynamically changes the CSS `display` property of the container for the radio buttons, showing or hiding them as needed.

In [23]:
def toggle_text_location_v3(widget, event, data):
    if data:
        v3_location_container.style_ = 'display: block; margin-top: 10px;'
    else:
        v3_location_container.style_ = 'display: none;'

#### 4.3.2 - Defining the search handler for UI #3

The `v3_on_search` function is triggered by the 'Search' button. It displays a loading indicator, calls the search logic, and then populates a `v.DataTable` with the results. To fix potential header overlap, a `<style>` tag is injected to prevent text wrapping in table headers. It also converts the pandas DataFrame to a JSON-compatible format.

In [24]:
v3_results_df = None

def v3_on_search(widget, event, data):
    global v3_results_df
    v3_results_card.children = [v.CardText(children=["🔍 Searching..."])]
    result = execute_combined_search(
        applicant=v3_applicant.v_model, tech_field=v3_tech_field.v_model,
        search_text=v3_text.v_model, text_location=v3_location.v_model, limit=int(v3_limit.v_model or 10)
    )
    if result["success"] and not result["data"].empty:
        v3_results_df = result["data"]
        
        
        display_df = v3_results_df.copy()
        if 'abstract' in display_df.columns:
            display_df['abstract'] = display_df['abstract'].str.slice(0, 100) + '...'
            
        all_headers = [
            {'text': 'Appln ID', 'value': 'appln_id'},
            {'text': 'Authority', 'value': 'filing_authority'},
            {'text': 'Filing Date', 'value': 'filing_date'},
            {'text': 'Applicant Name', 'value': 'applicant_name'},
            {'text': 'Tech Sector', 'value': 'techn_sector'},
            {'text': 'Tech Field', 'value': 'techn_field'},
            {'text': 'Title', 'value': 'title'},
            {'text': 'Abstract', 'value': 'abstract'},
        ]
        
        # Filter headers to only show columns that actually exist in the final dataframe
        headers = [h for h in all_headers if h['value'] in display_df.columns]
        
        items = json.loads(display_df.to_json(orient='records', date_format='iso'))
        
        style_fix = v.Html(tag='style', children=['th { white-space: nowrap; }'])
        
        v3_results_card.children = [style_fix, v.CardTitle(children=["✅ Search Results"]), v.DataTable(headers=headers, items=items)]
        v3_export_btn.disabled = False
    else:
        v3_results_df = None
        v3_results_card.children = [v.Alert(type='warning', children=[result['message']])]
        v3_export_btn.disabled = True

#### 4.3.3 - Defining the export handler for UI #3

This `v3_on_export` function is connected to the 'Export' button. It saves the results to a CSV file and adds a temporary `Alert` message to the top of the results card to inform the user of the successful export.

In [25]:
def v3_on_export(widget, event, data):
    if v3_results_df is not None:
        filename = f"vuetify_search_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        v3_results_df.to_csv(filename, index=False)
        v3_results_card.children = [v.Alert(type='success', children=[f"💾 Exported to {filename}"])] + v3_results_card.children

#### 4.3.4 - Building and assembling the `ipyvuetify` interface

This cell constructs our third and most advanced UI. It defines all the `v.` components and connects their events to the handler functions. The layout is managed with `v.Container`, `v.Card`, `v.Row`, and `v.Col` for a responsive, modern design.

In [26]:
print("Building user interface #3: ipyvuetify (Material Design)...")

v3_applicant = v.TextField(label="Company Name", v_model="", outlined=True, clearable=True)
v3_tech_field = v.TextField(label="Technology Field", v_model="", outlined=True, clearable=True)
v3_text = v.TextField(label="Text Search", v_model="", outlined=True, clearable=True)
v3_location = v.RadioGroup(v_model='title', children=[v.Radio(label=l.title(), value=l) for l in ['title', 'abstract', 'both']], row=True)

v3_location_container = v.Container(children=[v.Html(tag='div', class_='font-weight-bold mb-2', children=['Search In:']), v3_location], style_='display: none;')
v3_limit = v.Slider(label="Max Results", v_model=10, min=5, max=50, step=5, thumb_label=True)
v3_search_btn = v.Btn(color='primary', children=['Search Patents'])
v3_export_btn = v.Btn(color='success', children=['Export CSV'], disabled=True)
v3_results_card = v.Card(children=[v.CardText(children=["Results will appear here."])], outlined=True, class_='mt-4')

v3_text.on_event('input', toggle_text_location_v3)
v3_search_btn.on_event('click', v3_on_search)
v3_export_btn.on_event('click', v3_on_export)

vuetify_interface = v.Container(children=[
    v.Card(class_='pa-4', children=[
        v.Html(tag='h3', children=['Interface #3: ipyvuetify'], class_='mb-4 text-center'),
        v3_applicant, v3_tech_field, v3_text, v3_location_container, v3_limit,
        v.Row(justify='center', class_='mt-2', children=[v.Col(cols='auto', children=[v3_search_btn]), v.Col(cols='auto', children=[v3_export_btn])])
    ]),
    v3_results_card
])

print("✅ The interface is ready. Run the next cell to display it.")

Building user interface #3: ipyvuetify (Material Design)...
✅ The interface is ready. Run the next cell to display it.


#### 4.3.5 - Displaying the `ipyvuetify` interface

This cell displays the `vuetify_interface` object. You will see the modern Material Design interface, ready for interaction.

In [27]:
display(vuetify_interface)

Container(children=[Card(children=[Html(children=['Interface #3: ipyvuetify'], class_='mb-4 text-center', layo…

## Tutorial summary

You have now completed this hands-on tutorial for PATSTAT in TIP querying and interface design.

### Topics covered

#### Search techniques:
-   **Targeted queries**: Writing SQL to search PATSTAT by applicant, text content, and high-level WIPO technology fields.
-   **Dynamic data**: Fetching human-readable labels from the database to enrich raw data.
-   **Flexible logic**: Building a single, dynamic function that intelligently combines multiple search filters.

#### UI development:
You now have a practical understanding of three distinct approaches to building interfaces in Jupyter:
1.  **`ipywidgets`**: A reliable choice for rapid prototyping and simple, effective internal tools.
2.  **`CSS`/`HTML`**: An advanced solution for fully custom, professional interfaces where visual control is paramount.
3.  **`ipyvuetify`**: A framework for building modern, feature-rich, and application-like experiences.

### Further applications

The skills learned here are directly applicable to building a variety of tools, including:
-   Corporate IP monitoring dashboards.
-   Technology trend analysis platforms.
-   Competitive intelligence tools.

You have progressed from basic queries to building interactive systems. You can now adapt and expand upon these examples for your own projects.